In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import statsmodels.api as sm

# Create output folders
os.makedirs("data", exist_ok=True)
os.makedirs("visuals", exist_ok=True)

# 1. LOAD & CLEAN DATA
# Kaggle Superstore CSV typically uses windows-1252 / latin1 encoding
df = pd.read_csv("Sample - Superstore.csv", encoding="windows-1252")

# Remove exact duplicates and standardize column names
df = df.drop_duplicates().copy()
df["Order Date"] = pd.to_datetime(df["Order Date"])
df["Ship Date"] = pd.to_datetime(df["Ship Date"])

# Feature Engineering
df["Order Year"] = df["Order Date"].dt.year
df["Order Month"] = df["Order Date"].dt.to_period("M").dt.to_timestamp()
df["Month Name"] = df["Order Date"].dt.strftime("%b")
df["Month Num"] = df["Order Date"].dt.month
df["Shipping Days"] = (df["Ship Date"] - df["Order Date"]).dt.days
df["Profit Margin (%)"] = np.where(df["Sales"] > 0, (df["Profit"] / df["Sales"]) * 100, 0)

# Save cleaned dataset
df.to_csv("data/superstore_sales_cleaned.csv", index=False)

# 2. PRINT EXECUTIVE KPI SUMMARY
total_sales = df["Sales"].sum()
total_profit = df["Profit"].sum()
overall_margin = (total_profit / total_sales) * 100
total_orders = df["Order ID"].nunique()
total_customers = df["Customer ID"].nunique()
aov = total_sales / total_orders

print("=" * 55)
print("EXECUTIVE KPI SUMMARY")
print("=" * 55)
print(f"Total Revenue:        ${total_sales:,.2f}")
print(f"Total Profit:         ${total_profit:,.2f}")
print(f"Overall Profit Margin:{overall_margin:.2f}%")
print(f"Total Unique Orders:  {total_orders:,}")
print(f"Total Customers:      {total_customers:,}")
print(f"Average Order Value:  ${aov:,.2f}")
print("=" * 55)

print("\n--- PERFORMANCE BY CATEGORY ---")
cat_summary = df.groupby("Category").agg(
    Revenue=("Sales", "sum"),
    Profit=("Profit", "sum"),
    Units=("Quantity", "sum"),
    Avg_Discount=("Discount", "mean")
)
cat_summary["Margin (%)"] = (cat_summary["Profit"] / cat_summary["Revenue"]) * 100
print(cat_summary.round(2))

print("\n--- PERFORMANCE BY REGION ---")
reg_summary = df.groupby("Region").agg(
    Revenue=("Sales", "sum"),
    Profit=("Profit", "sum"),
    Orders=("Order ID", "nunique")
)
reg_summary["Margin (%)"] = (reg_summary["Profit"] / reg_summary["Revenue"]) * 100
print(reg_summary.sort_values(by="Revenue", ascending=False).round(2))

print("\n--- BOTTOM 5 LOSS-MAKING SUB-CATEGORIES ---")
subcat_profit = df.groupby("Sub-Category")[["Sales", "Profit"]].sum().sort_values(by="Profit")
print(subcat_profit.head(5).round(2))

# 3. STATISTICAL ANALYSIS: IMPACT OF DISCOUNT ON PROFIT MARGIN (OLS)
X = sm.add_constant(df["Discount"])
y = df["Profit Margin (%)"]
ols_model = sm.OLS(y, X).fit()
print("\n--- OLS REGRESSION: DISCOUNT VS. PROFIT MARGIN (%) ---")
print(f"Intercept (0% Discount Margin): {ols_model.params['const']:.2f}%")
print(f"Slope (Per +10% Discount):      {ols_model.params['Discount']*0.10:.2f}% margin change")
print(f"R-squared:                      {ols_model.rsquared:.4f} (p-value: {ols_model.pvalues['Discount']:.2e})")

# 4. GENERATE CLIENT-READY VISUALIZATIONS
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["font.sans-serif"] = "DejaVu Sans"

# CHART 1: Monthly Revenue & Profit Trend
monthly = df.groupby("Order Month")[["Sales", "Profit"]].sum().reset_index()
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(monthly["Order Month"], monthly["Sales"], color="#1f4e78", linewidth=2.2, label="Monthly Revenue")
ax.plot(monthly["Order Month"], monthly["Profit"], color="#2e8b57", linewidth=2.2, label="Monthly Profit")
ax.fill_between(monthly["Order Month"], monthly["Sales"], color="#1f4e78", alpha=0.08)
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter("${x:,.0f}"))
ax.set_title("Monthly Revenue vs. Profit Trend (2014–2017)", fontsize=13, fontweight="bold", pad=12)
ax.set_ylabel("USD ($)")
ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig("visuals/01_monthly_revenue_profit_trend.png", dpi=300)
plt.close()

# CHART 2: Category & Regional Revenue vs. Profit
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cat_plot = cat_summary.reset_index().sort_values("Revenue", ascending=False)
x = np.arange(len(cat_plot))
width = 0.36
axes[0].bar(x - width/2, cat_plot["Revenue"], width, label="Revenue", color="#1f4e78")
axes[0].bar(x + width/2, cat_plot["Profit"], width, label="Profit", color="#2e8b57")
axes[0].set_xticks(x)
axes[0].set_xticklabels(cat_plot["Category"])
axes[0].yaxis.set_major_formatter(mtick.StrMethodFormatter("${x:,.0f}"))
axes[0].set_title("Revenue & Profit by Category", fontsize=12, fontweight="bold")
axes[0].grid(axis="y", linestyle="--", alpha=0.4)
axes[0].legend(frameon=False)

reg_plot = reg_summary.reset_index().sort_values("Revenue", ascending=False)
x2 = np.arange(len(reg_plot))
axes[1].bar(x2 - width/2, reg_plot["Revenue"], width, label="Revenue", color="#1f4e78")
axes[1].bar(x2 + width/2, reg_plot["Profit"], width, label="Profit", color="#2e8b57")
axes[1].set_xticks(x2)
axes[1].set_xticklabels(reg_plot["Region"])
axes[1].yaxis.set_major_formatter(mtick.StrMethodFormatter("${x:,.0f}"))
axes[1].set_title("Revenue & Profit by Region", fontsize=12, fontweight="bold")
axes[1].grid(axis="y", linestyle="--", alpha=0.4)
axes[1].legend(frameon=False)

plt.tight_layout()
plt.savefig("visuals/02_category_regional_performance.png", dpi=300)
plt.close()

# CHART 3: Top 10 Products by Revenue
top_products = (
    df.groupby("Product Name")["Sales"]
    .sum()
    .sort_values(ascending=True)
    .tail(10)
)
short_names = [name[:42] + "..." if len(name) > 42 else name for name in top_products.index]
fig, ax = plt.subplots(figsize=(10, 5.5))
bars = ax.barh(short_names, top_products.values, color="#2b5c8f")
ax.xaxis.set_major_formatter(mtick.StrMethodFormatter("${x:,.0f}"))
ax.set_title("Top 10 Revenue-Generating Products", fontsize=13, fontweight="bold", pad=12)
ax.set_xlabel("Total Revenue (USD)")
ax.grid(axis="x", linestyle="--", alpha=0.4)
for bar in bars:
    w = bar.get_width()
    ax.text(w + 800, bar.get_y() + bar.get_height()/2, f"${w:,.0f}", va="center", fontsize=8.5)
plt.xlim(0, top_products.max() * 1.15)
plt.tight_layout()
plt.savefig("visuals/03_top_10_products_revenue.png", dpi=300)
plt.close()

# CHART 4: Discount Tiers vs. Average Profit Margin
df["Discount Tier"] = pd.cut(
    df["Discount"],
    bins=[-0.01, 0.0, 0.20, 0.40, 1.0],
    labels=["0% (No Discount)", "1%–20% (Low)", "21%–40% (Moderate)", ">40% (Heavy)"]
)
tier_margin = df.groupby("Discount Tier", observed=False)["Profit Margin (%)"].mean()
colors = ["#2e8b57", "#4a90d9", "#e68a00", "#c0392b"]
fig, ax = plt.subplots(figsize=(9, 4.8))
bars = ax.bar(tier_margin.index, tier_margin.values, color=colors, width=0.55)
ax.axhline(0, color="black", linewidth=0.9)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_title("Average Profit Margin (%) by Discount Tier", fontsize=13, fontweight="bold", pad=12)
ax.set_ylabel("Mean Profit Margin (%)")
ax.grid(axis="y", linestyle="--", alpha=0.4)
for bar in bars:
    h = bar.get_height()
    offset = 2.5 if h >= 0 else -5.5
    ax.text(bar.get_x() + bar.get_width()/2, h + offset, f"{h:.1f}%", ha="center", fontweight="bold", fontsize=9.5)
plt.ylim(min(tier_margin.values) - 12, max(tier_margin.values) + 10)
plt.tight_layout()
plt.savefig("visuals/04_discount_vs_profit_margin.png", dpi=300)
plt.close()

print("\nAll 4 charts saved successfully to visuals/ and cleaned CSV saved to data/!")

EXECUTIVE KPI SUMMARY
Total Revenue:        $2,297,200.86
Total Profit:         $286,397.02
Overall Profit Margin:12.47%
Total Unique Orders:  5,009
Total Customers:      793
Average Order Value:  $458.61

--- PERFORMANCE BY CATEGORY ---
                   Revenue     Profit  Units  Avg_Discount  Margin (%)
Category                                                              
Furniture        741999.80   18451.27   8028          0.17        2.49
Office Supplies  719047.03  122490.80  22906          0.16       17.04
Technology       836154.03  145454.95   6939          0.13       17.40

--- PERFORMANCE BY REGION ---
           Revenue     Profit  Orders  Margin (%)
Region                                           
West     725457.82  108418.45    1611       14.94
East     678781.24   91522.78    1401       13.48
Central  501239.89   39706.36    1175        7.92
South    391721.90   46749.43     822       11.93

--- BOTTOM 5 LOSS-MAKING SUB-CATEGORIES ---
                  Sales    Prof